# 40 – TOC / TOD Geometries Cleaning

This notebook prepares **Transit-Oriented Communities (TOCs)** and **TOD / station-area** geometries.

**Goals:**
- Load raw TOC and TOD shapefiles.
- Check / set CRS and reproject to the **project CRS**.
- Keep only essential fields (IDs, names, geometry).
- Save cleaned TOC/TOD layers for spatial analysis.

In [12]:
from pathlib import Path
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
BOUNDARIES_DIR = DATA_DIR / "boundaries"

BOUNDARIES_DIR

WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries')

In [13]:
# TODO: update filenames
raw_toc_path = BOUNDARIES_DIR / "tocs" / "Walkable_Urban_Code.shp"

toc_clean_path = BOUNDARIES_DIR / "processed" / "toc_clean.gpkg"

raw_toc_path, toc_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/tocs/Walkable_Urban_Code.shp'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'))

In [14]:
toc = gpd.read_file(raw_toc_path)

print("TOC features:", len(toc), "| CRS:", toc.crs)

toc.head()

TOC features: 11 | CRS: EPSG:2868


,OBJECTID,APPLICABIL,geometry
0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."


In [15]:
PROJECT_CRS = "EPSG:2868"

def ensure_project_crs(gdf, name="layer"):
    if gdf.crs is None:
        print(f"{name}: CRS is None – set manually before reprojecting.")
        # Example:
        # gdf = gdf.set_crs("EPSG:####")
        return gdf
    
    if PROJECT_CRS is not None and gdf.crs != PROJECT_CRS:
        print(f"{name}: Reprojecting from {gdf.crs} to {PROJECT_CRS}")
        return gdf.to_crs(PROJECT_CRS)
    else:
        print(f"{name}: No reprojection performed.")
        return gdf

toc = ensure_project_crs(toc, "TOC")

TOC: No reprojection performed.


In [16]:
toc.columns

Index(['OBJECTID', 'APPLICABIL', 'geometry'], dtype='object')

In [17]:
TOC_ID_FIELD = "OBJECTID"      # e.g. "TOC_ID"
TOC_NAME_FIELD = "APPLICABIL"  # e.g. "TOC_NAME"

toc_clean = toc[[TOC_ID_FIELD, TOC_NAME_FIELD, "geometry"]].copy()

toc_clean.head()

,OBJECTID,APPLICABIL,geometry
0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."


In [18]:
toc_clean.to_file(toc_clean_path, driver="GPKG")

print("Saved cleaned TOC:", toc_clean_path)

Saved cleaned TOC: c:\Users\arjav\DevSource\toc-performance-dashboard\data\boundaries\processed\toc_clean.gpkg
